# Corrected WP release verification

## tl;dr

The frozen rule selects shared lambda 3000. Later-period RMSE improves from 15.366597 to 14.703783 versus corrected lambda 150000. All five later seasons improve. This notebook reproduces the decision from saved predictions without fitting new models.

## Context & Methods

The shared offense/defense penalty is selected using 2016–2021 outcomes. A frozen later-period check uses 2022–2026. The only fallback is 150,000. Each margin calibration uses earlier outcomes only. The probability clip stays at 2.5%.

### Key Assumptions

These are reused historical outcomes with observed future lineups. Bootstrap uncertainty is conditional on fitted models. The historical score inputs are cumulative possession-point proxies. Legacy caches through 2023 omit overtime possessions, so missing overtime change enters the last regulation lineup's terminal credit. Official final scores supply winner labels and evaluation margins. Season 2027 is excluded. Published raw WP uses five-year windows; published logit WP uses one-year windows.


## Data

Inputs are the pinned run manifest, its saved predictions, the frozen experiment contract, and published aggregate rating files. Run from the New SPM repository using its Python environment.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from nba_impact.data.manifest import sha256_file
from research.rapm_lab.run_wp_chronology_release import select_lambda, publication_gate

root = Path.cwd()
assert (root / 'research/experiments/wp_chronology_release_v2.json').exists()
run_id = 'wp_chronology_release_v2_e3a3fbf4c2'
output = root / 'research/rapm_lab/outputs/wp_chronology_release' / run_id
run = json.loads((output / 'run.json').read_text())
for name, digest in run['artifact_hashes'].items():
    assert sha256_file(output / name) == digest, name
for name, digest in run['input_hashes'].items():
    assert sha256_file(root / name) == digest, name
predictions = pd.read_parquet(output / 'game_predictions.parquet')
assert predictions.outcome_season.max() == 2026
assert not predictions.duplicated(['candidate', 'outcome_season', 'game_id']).any()
print('All pinned input and artifact hashes match. No 2027 outcomes.')


All pinned input and artifact hashes match. No 2027 outcomes.


## Results

### Reproduce the selection and publication gate


In [2]:
selection = select_lambda(predictions, run['config'])
gate = publication_gate(predictions, selection['selected_lambda'], run['config'])
assert selection == run['selection']
assert gate == run['publication_gate']
print(json.dumps(gate, indent=2))
pd.DataFrame(selection['candidates'])


{
  "passed": true,
  "published_lambda": 3000,
  "rmse_delta": -0.6628140089984704,
  "upper_rmse_delta": -0.5852699272289618,
  "maximum_season_rmse_delta": -0.3133960857330642
}


,lambda,rmse,excess_rmse,simultaneous_upper_excess,eligible
0,100,13.449654,0.136944,0.245244,False
1,300,13.389961,0.077251,0.185552,False
2,1000,13.324487,0.011777,0.120078,False
3,3000,13.312710,0.000000,0.000000,True
4,10000,13.369993,0.057283,0.165584,False
5,30000,13.435478,0.122767,0.231068,False
6,150000,13.748407,0.435697,0.543998,False


### Independently recompute the later-period benchmark

Square root of the equally weighted season mean squared errors. The target is official final game margin, including technical free throws.


In [3]:
later = predictions.loc[predictions.outcome_season.isin(run['config']['diagnostic_outcomes'])].copy()
later['squared_error'] = (later.actual_margin - later.predicted_margin) ** 2
summary = later.groupby(['candidate', 'outcome_season']).squared_error.mean().groupby('candidate').mean().pow(.5)
for row in run['benchmark_summary']:
    assert np.isclose(summary[row['candidate']], row['equal_season_rmse'], rtol=0, atol=1e-12)
labels = {'pulse': 'PULSE', 'rapm': 'RAPM', 'raw_wp': 'Raw WP', f"logit_{gate['published_lambda']}": 'Log-odds WP'}
summary.loc[list(labels)].rename(index=labels).rename('Official margin RMSE').to_frame()


,Official margin RMSE
candidate,
PULSE,14.441775
RAPM,14.599081
Raw WP,14.934896
Log-odds WP,14.703783


### Check the public rating windows and totals


In [4]:
for filename, window in [('public_logit_ratings.parquet', 1), ('public_raw_rolling_ratings.parquet', 5)]:
    frame = pd.read_parquet(output / filename)
    assert set(frame.Season) == {2024, 2025, 2026}
    assert frame.window_end.eq(frame.Season).all()
    assert (frame.window_end - frame.window_start + 1).eq(window).all()
    assert frame.player_name.notna().all()
    assert frame[['off_possessions', 'def_possessions']].min(axis=1).gt(0).all()
    np.testing.assert_allclose(frame.net_per_100, frame.offense_per_100 + frame.defense_per_100, atol=1e-12)
    print(filename, 'passes window, name, exposure, and additive-total checks')
print('Maximum game conservation error:', run['quality']['maximum_conservation_error'])


public_logit_ratings.parquet passes window, name, exposure, and additive-total checks
public_raw_rolling_ratings.parquet passes window, name, exposure, and additive-total checks
Maximum game conservation error: 1.3322676295501878e-15


## Takeaways

Use the published penalty reported by the frozen gate above. Do not select another penalty using later-period results. Predictive equivalence after affine calibration does not establish that larger player coefficients are more accurate. This release repairs the descriptive WP boards; it does not promote WP over PULSE or replace the frozen PULSE model.
